# Traivo Attraction Recommender
A small KNN model using the same DictVectorizer and pickle approach as the destination model. The attraction rows are a curated starter dataset; recommendation labels are generated from simple interest, budget, and time rules.

In [ ]:
from pathlib import Path
import pandas as pd
from itertools import product
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import pickle

model_dir = Path.cwd()
data_path = model_dir.parent / 'data_processing' / 'traivo_attractions.csv'
attractions = pd.read_csv(data_path)
attractions.head()

In [ ]:
training_rows = []
for _, place in attractions.iterrows():
    for vibe, budget, hours in product(
        ['any', 'beach', 'nature', 'history', 'culture', 'adventure', 'food'],
        ['any', 'low', 'medium', 'high'], [2, 4, 8]
    ):
        cost_match = budget == 'any' or place['cost'] == 'low' or (budget == 'medium' and place['cost'] == 'medium') or budget == 'high'
        vibe_match = vibe == 'any' or place['vibe'] == vibe
        enough_time = int(place['visit_hours']) <= hours
        training_rows.append({
            'category': place['category'], 'vibe': place['vibe'], 'cost': place['cost'],
            'visit_hours': int(place['visit_hours']), 'vibe_preference': vibe,
            'budget_preference': budget, 'hours_available': hours,
            'recommended': int(cost_match and vibe_match and enough_time)
        })
training_data = pd.DataFrame(training_rows)
training_data.head()

In [ ]:
feature_columns = ['category', 'vibe', 'cost', 'visit_hours', 'vibe_preference', 'budget_preference', 'hours_available']
vectorizer = DictVectorizer()
features = vectorizer.fit_transform(training_data[feature_columns].to_dict('records'))
labels = training_data['recommended']
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42, stratify=labels)
model = KNeighborsClassifier(n_neighbors=5, weights='distance')
model.fit(X_train, y_train)
print('Accuracy:', accuracy_score(y_test, model.predict(X_test)))

In [ ]:
with open(model_dir / 'attraction_model.pkl', 'wb') as model_file:
    pickle.dump((model, vectorizer), model_file)

print('Saved attraction_model.pkl')